# grid2benchmark Tutorial

This notebook walks you through the full workflow of the **grid2benchmark** package — a standalone benchmarking harness for [Grid2Op](https://grid2op.readthedocs.io/) power-system reinforcement-learning algorithms.

**What you'll learn:**
1. Installation and setup
2. Importing the package
3. Core features and basic usage (Python API + CLI)
4. Configuration options (`BenchmarkConfig`, `ScenarioConfig`)
5. Handling input and output (topology / time-series files, result structure)
6. Error handling and debugging
7. Advanced usage — multiple scenarios, backend selection, custom agents

## 1. Installation and Setup

Install `grid2benchmark` and its dependencies with `uv` (recommended) or `pip`:

```bash
# Recommended: uv
uv add grid2benchmark

# Or pip
pip install grid2benchmark
```

> **Note:** `grid2op`, `pandapower`, `lightsim2grid`, and `pypowsybl2grid` are required dependencies installed automatically.

Verify the installation:

In [ ]:
import importlib.metadata
print("grid2benchmark version:", importlib.metadata.version("grid2benchmark"))

import grid2op
print("grid2op       version:", grid2op.__version__)

import pandapower as pp
print("pandapower    version:", pp.__version__)

## 2. Importing the Package

The main public API is available from the top-level `grid2benchmark` module:

In [ ]:
from pathlib import Path

from grid2benchmark import (
    run_benchmark,       # high-level orchestration function
    BenchmarkConfig,     # top-level config (max steps, KPIs, scenarios)
    ScenarioConfig,      # per-scenario config (env, topology, time-series, backend)
    TopologySource,      # points to an external grid topology file
    TimeSeriesSource,    # points to an external chronics directory
)

print("Imports OK")

## 3. Core Features and Basic Usage

### 3.1 Minimal benchmark run (built-in Grid2Op environment)

The simplest call uses Grid2Op's bundled `l2rpn_case14_sandbox` environment,
runs the first time-series episode for 10 steps, and evaluates three KPIs.

The `algorithm` argument can be:
- a `Path` to a `.py` file containing a `build_agent(env, context)` function, or
- a Python source-code string with the same function.

In [ ]:
ALGORITHM_SOURCE = """
class DoNothingAgent:
    def __init__(self, action_space):
        self._as = action_space

    def act(self, observation):
        return self._as()          # no-op action

def build_agent(env, context):
    return DoNothingAgent(env.action_space)
"""

result = run_benchmark(
    ALGORITHM_SOURCE,
    BenchmarkConfig(
        scenarios=[
            ScenarioConfig(
                env_name="l2rpn_case14_sandbox",
                time_series_ids=(0,),   # run only the first episode
            )
        ],
        max_steps=10,
        kpis=("survival", "violations", "latency"),
    ),
)

print("Top-level keys:", list(result.keys()))
print("Scenario count:", result["summary"]["scenario_count"])
print("Episode count :", result["summary"]["episode_count"])

### 3.2 Running from a file (the template algorithm)

The `examples/` directory ships two ready-to-use algorithms.
Pass a `Path` object to load an algorithm from disk:

In [ ]:
result_file = run_benchmark(
    Path("examples/algorithm_template.py"),
    BenchmarkConfig(
        scenarios=[ScenarioConfig(time_series_ids=(0,))],
        max_steps=5,
        kpis=("survival", "latency"),
    ),
)

# Episode detail from the first scenario
ep = result_file["scenarios"][0]["episodes"][0]
print(f"Steps completed : {ep['steps']}")
print(f"Runtime (s)     : {ep['runtime_seconds']:.4f}")
print(f"Terminated      : {ep['terminated']}")

### 3.3 CLI quick-start

You can drive benchmarks from a terminal without writing any Python:

```bash
grid2benchmark run \
  --algorithm examples/algorithm_template.py \
  --env l2rpn_case14_sandbox \
  --time-series 0 \
  --max-steps 10 \
  --kpis survival,violations,latency \
  --output results.json
```

The `results.json` file has the same structure as the Python API result.

## 4. Working with Configuration Options

### 4.1 `BenchmarkConfig`

| Field | Type | Default | Description |
|---|---|---|---|
| `scenarios` | `tuple[ScenarioConfig, ...]` | one default scenario | Scenario list |
| `max_steps` | `int` | `200` | Max steps per episode |
| `kpis` | `tuple[str, ...]` | all available | KPIs to compute |

Available KPI names:

In [ ]:
from grid2benchmark._config import AVAILABLE_KPI_NAMES, DEFAULT_MAX_STEPS

print("Available KPIs:")
for name in AVAILABLE_KPI_NAMES:
    print(" •", name)

print(f"\nDefault max steps: {DEFAULT_MAX_STEPS}")

### 4.2 `ScenarioConfig`

| Field | Type | Default | Description |
|---|---|---|---|
| `env_name` | `str` | `l2rpn_case14_sandbox` | Grid2Op environment name |
| `time_series_ids` | `tuple[int, ...] \| None` | `None` (all) | Which episodes to run |
| `topology` | `TopologySource \| None` | `None` | External topology file |
| `time_series` | `TimeSeriesSource \| None` | `None` | External chronics dir |
| `backend` | `str \| None` | `None` | Power-flow solver: `pandapower`, `lightsim2grid`, or `pypowsybl` |

In [ ]:
from grid2benchmark._config import SUPPORTED_BACKENDS

print("Supported backends:", SUPPORTED_BACKENDS)

# Selecting a subset of time-series episodes
sc = ScenarioConfig(
    env_name="l2rpn_case14_sandbox",
    time_series_ids=(0, 2, 4),   # only episodes 0, 2, and 4
)
print(f"\nScenario env  : {sc.env_name}")
print(f"Episode IDs   : {sc.time_series_ids}")
print(f"Backend       : {sc.backend!r}  (None = auto)")

### 4.3 Backend selection

When `backend` is `None` (the default):
- If a `topology` file is provided → `pandapower` is used automatically.
- If no topology is provided → Grid2Op's built-in default applies.

```python
ScenarioConfig(backend="pandapower")    # always use PandaPower
ScenarioConfig(backend="lightsim2grid") # fast C++ solver (requires lightsim2grid)
ScenarioConfig(backend="pypowsybl")     # PowerSyBl solver (requires pypowsybl2grid)
```

> If a backend package is not installed, a clear `ImportError` with an install command
> is raised **at runtime** — never at import time.

## 5. Handling Input and Output

### 5.1 External topology and time-series files

Provide your own pandapower grid JSON and a Grid2Op chronics directory instead of relying on a built-in environment:

In [ ]:
TOPOLOGY_FILE = Path("test_data/rte_case118_example/grid.json")
CHRONICS_DIR  = Path("test_data/rte_case118_example/chronics")

if TOPOLOGY_FILE.exists() and CHRONICS_DIR.exists():
    sc_external = ScenarioConfig(
        env_name="rte_case118_example",
        topology=TopologySource(
            format="pandapower",
            path=TOPOLOGY_FILE,
        ),
        time_series=TimeSeriesSource(
            format="grid2op_chronics_dir",
            path=CHRONICS_DIR,
        ),
        time_series_ids=(0,),
    )
    print("ScenarioConfig with external files:")
    print(f"  topology : {sc_external.topology.path}")
    print(f"  chronics : {sc_external.time_series.path}")
else:
    print("test_data not found — skipping.")
    print("Populate it by copying the bundled dataset:")
    print("  Copy-Item -Recurse .venv/Lib/site-packages/grid2op/data/rte_case118_example test_data/")

### 5.2 Scenario JSON file format (for `--scenarios`)

```json
[
    {
        "env_name": "l2rpn_case14_sandbox",
        "time_series_ids": [0, 1],
        "backend": "pandapower"
    },
    {
        "env_name": "rte_case118_example",
        "topology": { "format": "pandapower", "path": "./data/grid.json" },
        "time_series": { "format": "grid2op_chronics_dir", "path": "./data/chronics" },
        "time_series_ids": [0],
        "backend": "lightsim2grid"
    }
]
```

```bash
grid2benchmark run --algorithm algo.py --scenarios scenarios.json --output results.json
```

### 5.3 Understanding the output structure

```
result
├── scenarios
│   └── [0]
│       ├── scenario_index
│       ├── environment           env_name, backend, topology, time_series
│       ├── executed_time_series_ids
│       ├── episodes
│       │   └── [0]               episode_index, time_series_id, steps,
│       │                         overload_violations, runtime_seconds, terminated
│       └── kpis                  aggregated metrics over all episodes
└── summary
    ├── scenario_count
    ├── episode_count
    └── kpis                      mean/min/max/count per KPI across all scenarios
```